# s17 — Interrupts and Cancellation

**What this teaches:** how to cancel a long-running agent turn cleanly. The example wires `signal.NotifyContext(SIGINT)` so that Ctrl-C at a CLI prompt unwinds the agent loop without leaking goroutines or leaving the model mid-stream. The mechanism is plain `context.Context` cancellation — the same primitive works for *any* cancellation source.

**Concept anchor:** every call into the agent (model invocations, tool dispatches, stream emission) is `ctx`-aware. Cancel the context and the loop unwinds at the next safe boundary. Whatever you wire to that cancel — a signal, a timeout, an HTTP client disconnect — looks the same to the runner.

## Prerequisites

- A working [GoNB](https://github.com/janpfeifer/gonb) kernel.
- Any configured LLM provider.
  ```
  export YOKE_PROVIDER=openai_compat
  export OPENAI_BASE_URL=http://localhost:11434/v1
  export YOKE_MODEL=qwen2.5-coder
  ```
- **Note on testing from a notebook.** The reference `main.go` uses `signal.NotifyContext(SIGINT)` — but a Jupyter kernel does not let you press Ctrl-C and have it land in your running Go code as a real `SIGINT`. We therefore demonstrate the *mechanism* (a cancellable context driven by an external trigger) with a variation that **is** testable: a goroutine that calls `cancel()` after a short delay.

## 1. Imports

We import `os/signal` for completeness — the SIGINT wiring still works if you copy this code into a standalone binary.

In [ ]:
import (
    "context"
    "fmt"
    "os"
    "os/signal"
    "time"

    "github.com/blouargant/yoke/core/agentkit"
    "github.com/blouargant/yoke/core/stream"
)

_ = signal.Interrupt // referenced below in the SIGINT variant

## 2. Helper

In [ ]:
func must(err error) {
    if err != nil {
        panic(fmt.Sprintf("%v", err))
    }
}

## 3. The original SIGINT pattern (for reference)

This is what the standalone `main.go` does. Don't run it as-is in the notebook — there is no way to deliver a real `SIGINT` to the kernel — but the shape is identical to the cancellable variant below.

```go
ctx, cancel := signal.NotifyContext(context.Background(), os.Interrupt)
defer cancel()
// ... build llm / agent / runner ...
if err := stream.Print(os.Stdout, agentkit.RunOnceStream(ctx, r, prompt)); err != nil {
    fmt.Fprintln(os.Stderr, "run ended:", err)
}
```

The single load-bearing detail is `ctx, cancel := signal.NotifyContext(...)` — once the OS signal fires, the same `cancel()` you'd call manually is triggered automatically.

## 4. A notebook-testable variant: cancel after a delay

We build a cancellable `ctx` ourselves and schedule a `cancel()` from a goroutine. Everything downstream is identical to the SIGINT version — the runner can't tell the difference between an OS signal and a manual cancel.

In [ ]:
ctx, cancel := context.WithCancel(context.Background())
defer cancel()
llm, err := agentkit.NewModel(ctx)
must(err)
a, err := agentkit.New(agentkit.AgentConfig{
    Name:        "s17_interrupt",
    Description: "Cancellable agent.",
    Model:       llm,
})
must(err)
r, err := agentkit.Runner("s17", a)
must(err)

## 5. Launch a long task and cancel it mid-stream

We ask for a 1000-word essay (plenty of stream events) and schedule a `cancel()` 1.5 seconds in. `RunOnceStream` returns a non-nil error after cancellation — we print it instead of panicking so you can see the unwinding work.

In [ ]:
go func() {
    time.Sleep(1500 * time.Millisecond)
    cancel()
}()
prompt := "Write a 1000-word essay about agentic systems."
if err := stream.Print(os.Stdout, agentkit.RunOnceStream(ctx, r, prompt)); err != nil {
    fmt.Fprintln(os.Stderr, "run ended:", err)
}

## What to look for

- The streamed essay stops mid-sentence and `run ended:` appears with a `context.Canceled` value. No goroutine leaks, no orphan HTTP request — the runner respects the context across every internal boundary.
- The same pattern is how the HTTP server in `server/` aborts a turn when the browser closes the SSE connection — pair this with [s18_events](../s18_events/s18_events.ipynb) to see how `run_end` is emitted on cancellation.

## Try it yourself

1. Replace `context.WithCancel` with `context.WithTimeout(ctx, 1*time.Second)` and drop the goroutine — same effect, different trigger.
2. Move the `cancel()` *before* `stream.Print` to confirm the loop never starts. Then move it *after* and verify the run completes normally.